# 1. Data Pre-processing (25%)


---






In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from pytorch_tabnet.pretraining import TabNetPretrainer
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -----------------------------
# 1. Load datasets
# -----------------------------
china_df = pd.read_csv("China_dataset.csv")
wqi_df = pd.read_csv("WQI.csv").dropna()

# -----------------------------
# 2. Feature mapping (PHẢI CÙNG SỐ LƯỢNG)
# -----------------------------
china_features = [
    'Ammonia (mg/l)',
    'Biochemical Oxygen Demand (mg/l)',
    'Dissolved Oxygen (mg/l)'
]

wqi_features = [
    'NH4+_N',
    'COD',
    'DO'
]

# -----------------------------
# 3. Scaling
# -----------------------------
scaler_china = StandardScaler()
china_X = scaler_china.fit_transform(china_df[china_features]).astype(np.float32)

scaler_wqi = StandardScaler()
wqi_X = scaler_wqi.fit_transform(wqi_df[wqi_features]).astype(np.float32)

wqi_y = wqi_df['WQI'].values.reshape(-1, 1)
target_scaler = MinMaxScaler()
wqi_y_scaled = target_scaler.fit_transform(wqi_y).astype(np.float32)

# -----------------------------
# 4. UNSUPERVISED PRETRAIN (China)
# -----------------------------
pretrainer = TabNetPretrainer()
pretrainer.fit(
    X_train=china_X,
    eval_set=[china_X],
    max_epochs=100,
    patience=20,
    batch_size=256,
    virtual_batch_size=128
)

# -----------------------------
# 5. TRANSFER + FINE-TUNE (WQI)
# -----------------------------
X_train, X_val, y_train, y_val = train_test_split(
    wqi_X, wqi_y_scaled, test_size=0.2, random_state=42
)

tabnet = TabNetRegressor()
tabnet.fit(
    X_train=X_train,
    y_train=y_train,
    eval_set=[(X_val, y_val)],
    from_unsupervised=pretrainer,
    max_epochs=200,
    patience=20,
    batch_size=16,
    virtual_batch_size=8
)

# -----------------------------
# 6. Evaluate
# -----------------------------
y_pred_scaled = tabnet.predict(X_val)
mse = mean_squared_error(y_val, y_pred_scaled)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, y_pred_scaled)
r2 = r2_score(y_val, y_pred_scaled)
print("MSE (scaled):", mse)
print("RMSE (scaled):", rmse)
print("MAE (scaled):", mae)
print("R^2:", r2)

C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.93401 | val_0_unsup_loss_numpy: 0.9271199703216553|  0:00:06s
epoch 1  | loss: 0.88728 | val_0_unsup_loss_numpy: 0.9649199843406677|  0:00:13s
epoch 2  | loss: 0.85375 | val_0_unsup_loss_numpy: 1.379140019416809|  0:00:20s
epoch 3  | loss: 0.86218 | val_0_unsup_loss_numpy: 1.196079969406128|  0:00:27s
epoch 4  | loss: 0.80766 | val_0_unsup_loss_numpy: 0.9290099740028381|  0:00:34s
epoch 5  | loss: 0.85759 | val_0_unsup_loss_numpy: 1.3557699918746948|  0:00:41s
epoch 6  | loss: 0.82559 | val_0_unsup_loss_numpy: 1.0857599973678589|  0:00:47s
epoch 7  | loss: 0.84777 | val_0_unsup_loss_numpy: 1.147420048713684|  0:00:54s
epoch 8  | loss: 0.84844 | val_0_unsup_loss_numpy: 1.3253599405288696|  0:01:02s
epoch 9  | loss: 0.85085 | val_0_unsup_loss_numpy: 0.9730600118637085|  0:01:09s
epoch 10 | loss: 0.82777 | val_0_unsup_loss_numpy: 0.9956600069999695|  0:01:16s
epoch 11 | loss: 0.85892 | val_0_unsup_loss_numpy: 1.1911699771881104|  0:01:23s
epoch 12 | loss: 0.88569 | val_

C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\abstract_model.py:248: UserWarning: Loading weights from unsupervised pretraining
  warnings.warn("Loading weights from unsupervised pretraining")


epoch 0  | loss: 0.28786 | val_0_mse: 0.05564 |  0:00:00s
epoch 1  | loss: 0.0721  | val_0_mse: 0.02765 |  0:00:00s
epoch 2  | loss: 0.04469 | val_0_mse: 0.03415 |  0:00:01s
epoch 3  | loss: 0.04126 | val_0_mse: 0.03863 |  0:00:01s
epoch 4  | loss: 0.0387  | val_0_mse: 0.03072 |  0:00:02s
epoch 5  | loss: 0.03053 | val_0_mse: 0.02761 |  0:00:02s
epoch 6  | loss: 0.03669 | val_0_mse: 0.02975 |  0:00:03s
epoch 7  | loss: 0.03295 | val_0_mse: 0.02519 |  0:00:03s
epoch 8  | loss: 0.04001 | val_0_mse: 0.02479 |  0:00:04s
epoch 9  | loss: 0.02576 | val_0_mse: 0.0279  |  0:00:04s
epoch 10 | loss: 0.02789 | val_0_mse: 0.03061 |  0:00:05s
epoch 11 | loss: 0.02819 | val_0_mse: 0.02581 |  0:00:05s
epoch 12 | loss: 0.03449 | val_0_mse: 0.02663 |  0:00:06s
epoch 13 | loss: 0.03213 | val_0_mse: 0.0321  |  0:00:06s
epoch 14 | loss: 0.03191 | val_0_mse: 0.02344 |  0:00:07s
epoch 15 | loss: 0.02819 | val_0_mse: 0.03364 |  0:00:07s
epoch 16 | loss: 0.03704 | val_0_mse: 0.04527 |  0:00:08s
epoch 17 | los

C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


MSE (scaled): 0.019266624003648758
RMSE (scaled): 0.13880426507729782
MAE (scaled): 0.10907912254333496
R^2: 0.7473549842834473


In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from pytorch_tabnet.pretraining import TabNetPretrainer
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -----------------------------
# 1. Load datasets
# -----------------------------
china_df = pd.read_csv("China_dataset.csv")
wqi_df = pd.read_csv("WQI.csv").dropna()

# -----------------------------
# 2. Feature mapping (PHẢI CÙNG SỐ LƯỢNG)
# -----------------------------
china_features = [
    'Ammonia (mg/l)',
    'Nitrate (mg/l)',
    'Biochemical Oxygen Demand (mg/l)',
    'Dissolved Oxygen (mg/l)'
]

wqi_features = [
    'NH4+_N',
    'NO2-_N',
    'COD',
    'DO'
]

# -----------------------------
# 3. Scaling
# -----------------------------
scaler_china = StandardScaler()
china_X = scaler_china.fit_transform(china_df[china_features]).astype(np.float32)

scaler_wqi = StandardScaler()
wqi_X = scaler_wqi.fit_transform(wqi_df[wqi_features]).astype(np.float32)

wqi_y = wqi_df['WQI'].values.reshape(-1, 1)
target_scaler = MinMaxScaler()
wqi_y_scaled = target_scaler.fit_transform(wqi_y).astype(np.float32)

# -----------------------------
# 4. UNSUPERVISED PRETRAIN (China)
# -----------------------------
pretrainer = TabNetPretrainer()
pretrainer.fit(
    X_train=china_X,
    eval_set=[china_X],
    max_epochs=100,
    patience=20,
    batch_size=256,
    virtual_batch_size=128
)

# -----------------------------
# 5. TRANSFER + FINE-TUNE (WQI)
# -----------------------------
X_train, X_val, y_train, y_val = train_test_split(
    wqi_X, wqi_y_scaled, test_size=0.2, random_state=42
)

tabnet = TabNetRegressor()
tabnet.fit(
    X_train=X_train,
    y_train=y_train,
    eval_set=[(X_val, y_val)],
    from_unsupervised=pretrainer,
    max_epochs=200,
    patience=20,
    batch_size=16,
    virtual_batch_size=8
)

# -----------------------------
# 6. Evaluate
# -----------------------------
y_pred_scaled = tabnet.predict(X_val)
mse = mean_squared_error(y_val, y_pred_scaled)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, y_pred_scaled)
r2 = r2_score(y_val, y_pred_scaled)
print("MSE (scaled):", mse)
print("RMSE (scaled):", rmse)
print("MAE (scaled):", mae)
print("R^2:", r2)

C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.05059 | val_0_unsup_loss_numpy: 0.8623700141906738|  0:00:06s
epoch 1  | loss: 0.90617 | val_0_unsup_loss_numpy: 0.881309986114502|  0:00:13s
epoch 2  | loss: 0.90093 | val_0_unsup_loss_numpy: 0.8096500039100647|  0:00:19s
epoch 3  | loss: 0.86904 | val_0_unsup_loss_numpy: 0.8207600116729736|  0:00:26s
epoch 4  | loss: 0.90056 | val_0_unsup_loss_numpy: 0.9182900190353394|  0:00:32s
epoch 5  | loss: 0.89463 | val_0_unsup_loss_numpy: 0.8665099740028381|  0:00:38s
epoch 6  | loss: 0.88068 | val_0_unsup_loss_numpy: 0.828000009059906|  0:00:45s
epoch 7  | loss: 0.86472 | val_0_unsup_loss_numpy: 0.8382300138473511|  0:00:52s
epoch 8  | loss: 0.88862 | val_0_unsup_loss_numpy: 0.8252999782562256|  0:00:59s
epoch 9  | loss: 0.88602 | val_0_unsup_loss_numpy: 0.8974900245666504|  0:01:06s
epoch 10 | loss: 0.85032 | val_0_unsup_loss_numpy: 0.8467699885368347|  0:01:12s
epoch 11 | loss: 0.86004 | val_0_unsup_loss_numpy: 0.8557900190353394|  0:01:19s
epoch 12 | loss: 0.9024  | val

C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\abstract_model.py:248: UserWarning: Loading weights from unsupervised pretraining
  warnings.warn("Loading weights from unsupervised pretraining")


epoch 0  | loss: 1.14306 | val_0_mse: 0.15054 |  0:00:00s
epoch 1  | loss: 0.06508 | val_0_mse: 0.15373 |  0:00:01s
epoch 2  | loss: 0.0497  | val_0_mse: 0.0652  |  0:00:01s
epoch 3  | loss: 0.02949 | val_0_mse: 0.03279 |  0:00:02s
epoch 4  | loss: 0.03499 | val_0_mse: 0.03272 |  0:00:02s
epoch 5  | loss: 0.03441 | val_0_mse: 0.03402 |  0:00:03s
epoch 6  | loss: 0.02548 | val_0_mse: 0.02541 |  0:00:03s
epoch 7  | loss: 0.04183 | val_0_mse: 0.02997 |  0:00:04s
epoch 8  | loss: 0.025   | val_0_mse: 0.03156 |  0:00:04s
epoch 9  | loss: 0.03222 | val_0_mse: 0.0269  |  0:00:04s
epoch 10 | loss: 0.025   | val_0_mse: 0.0253  |  0:00:05s
epoch 11 | loss: 0.03053 | val_0_mse: 0.0439  |  0:00:05s
epoch 12 | loss: 0.03699 | val_0_mse: 0.05088 |  0:00:06s
epoch 13 | loss: 0.03303 | val_0_mse: 0.03264 |  0:00:06s
epoch 14 | loss: 0.02434 | val_0_mse: 0.02775 |  0:00:07s
epoch 15 | loss: 0.02813 | val_0_mse: 0.0214  |  0:00:07s
epoch 16 | loss: 0.02631 | val_0_mse: 0.02314 |  0:00:08s
epoch 17 | los

C:\Users\Admin\AppData\Local\miniconda3\envs\tabnet\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


MSE (scaled): 0.013194533996284008
RMSE (scaled): 0.11486746273982032
MAE (scaled): 0.09615994244813919
R^2: 0.8269789218902588
